In [1]:
import pandas as pd

In [3]:
# -------------------- Load Dataset --------------------
df = pd.read_csv("weather.csv")

# Attribute names (excluding target class)
attributes = list(df.columns[:-1])
target_attribute = df.columns[-1]

# -------------------- Find-S Algorithm --------------------
def find_s_algorithm(data):
    hypothesis = ['0'] * len(attributes)

    for _, row in data.iterrows():
        if row[target_attribute] == 'Yes':      # Consider only positive examples
            instance = list(row[:-1])

            if hypothesis == ['0'] * len(attributes):
                hypothesis = instance
            else:
                for i in range(len(hypothesis)):
                    if hypothesis[i] != instance[i]:
                        hypothesis[i] = '?'

    return hypothesis


# -------------------- Candidate Elimination Algorithm --------------------
def candidate_elimination(data):

    def is_consistent(hypothesis, instance):
        """Checks whether a hypothesis covers an instance."""
        for h, x in zip(hypothesis, instance):
            if h == '?':
                continue
            if h == '0':
                return False
            if h != x:
                return False
        return True

    # Initial boundaries
    S = [['0'] * len(attributes)]
    G = [['?'] * len(attributes)]

    for _, row in data.iterrows():
        instance = list(row[:-1])
        label = row[target_attribute]

        # ---------------- Positive Example ----------------
        if label == 'Yes':

            # Remove inconsistent hypotheses from G
            G = [g for g in G if is_consistent(g, instance)]

            # Generalize S
            new_S = []
            for s in S:
                s = s.copy()
                for i in range(len(s)):
                    if s[i] == '0':
                        s[i] = instance[i]
                    elif s[i] != instance[i]:
                        s[i] = '?'
                new_S.append(s)
            S = new_S

        # ---------------- Negative Example ----------------
        else:
            new_G = []

            for g in G:
                if is_consistent(g, instance):
                    # Specialize G
                    for i in range(len(g)):
                        if g[i] == '?':
                            for value in df[attributes[i]].unique():
                                if value != instance[i]:
                                    new_hypothesis = g.copy()
                                    new_hypothesis[i] = value

                                    # Keep only hypotheses more general than S
                                    for s in S:
                                        valid = True
                                        for h, sv in zip(new_hypothesis, s):
                                            if h != '?' and sv != '0' and h != sv:
                                                valid = False
                                                break
                                        if valid:
                                            new_G.append(new_hypothesis)
                else:
                    new_G.append(g)

            # Remove duplicate hypotheses
            G = []
            for h in new_G:
                if h not in G:
                    G.append(h)

    return S, G


# -------------------- Run Algorithms --------------------
print("Dataset Loaded:\n")
print(df)

# Find-S
find_s = find_s_algorithm(df)
print("\n========== FIND-S ==========")
print("Final Hypothesis:")
print(find_s)

# Candidate Elimination
S_final, G_final = candidate_elimination(df)

print("\n========== CANDIDATE ELIMINATION ==========")
print("\nSpecific Boundary (S):")
for s in S_final:
    print(s)

print("\nGeneral Boundary (G):")
for g in G_final:
    print(g)

Dataset Loaded:

     MinTemp  MaxTemp  Rainfall  Evaporation  Sunshine WindGustDir  \
0        8.0     24.3       0.0          3.4       6.3          NW   
1       14.0     26.9       3.6          4.4       9.7         ENE   
2       13.7     23.4       3.6          5.8       3.3          NW   
3       13.3     15.5      39.8          7.2       9.1          NW   
4        7.6     16.1       2.8          5.6      10.6         SSE   
..       ...      ...       ...          ...       ...         ...   
361      9.0     30.7       0.0          7.6      12.1         NNW   
362      7.1     28.4       0.0         11.6      12.7           N   
363     12.5     19.9       0.0          8.4       5.3         ESE   
364     12.5     26.9       0.0          5.0       7.1          NW   
365     12.3     30.2       0.0          6.0      12.6          NW   

     WindGustSpeed WindDir9am WindDir3pm  WindSpeed9am  ...  Humidity3pm  \
0             30.0         SW         NW           6.0  ...       